# PPMI imaging metadata table

Builds the per-image metadata table BIDS conversion depends on:
`mri_prep.bids.convert.convert_to_bids` looks up each raw image by `Image ID`
in this table and reads `PATNO`/`EVENT_ID`/`BIDS_Modality` (+ whatever
`PPMIAdapter.build_bids_name` needs, e.g. `Advanced_Modality`,
`Acquisition Plane`) off the matching row to decide the BIDS filename.

This notebook is the critical, must-work step referenced as "Processing
images" step 2 in the README -- keep it linear and focused on producing that
one CSV. Broader exploratory analysis (age/sex distributions, longitudinal
retention, UpSet plots, ...) lives in `ppmi_cohort_exploration.ipynb`
instead, since nothing downstream depends on it.

Steps: load + curate the ida.loni search export -> load clinical data ->
filter to usable image rows -> join -> save to `extra.merged_csv`.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

from mri_prep.config import REPO_ROOT, load_dataset_config
from mri_prep.datasets.ppmi import PPMIAdapter, reload_description_resources
from mri_prep.tabular.ida import describe_sequences, filter_images, load_ida_search, summarize

config = load_dataset_config("ppmi")
adapter = PPMIAdapter()

# ida.loni Advanced Search export -- see README "Selecting a cohort" step 1.
# Not part of DatasetPaths (it's a one-off input, not a pipeline stage
# output), so it's found directly under data/<DATASET>/search/.
search_csvs = sorted((REPO_ROOT / "data" / "PPMI" / "search").glob("idaSearch_*.csv"))
assert search_csvs, "No idaSearch_*.csv found under data/PPMI/search/"
search_csv = search_csvs[-1]  # most recent export
print(f"Using search export: {search_csv.name}")

Using search export: idaSearch_9_10_2026.csv


## 1. Load and curate the ida.loni search export

`annotate_categories` adds `category`/`ignored` (from the curated
`ppmi_description_categories.json` / `ppmi_ignored_descriptions.csv`) plus
the `BIDS_Modality`/`Advanced_Modality` split `build_bids_name` needs.

In [2]:
df = load_ida_search(search_csv)
df = adapter.annotate_categories(df)
print(f"Loaded {len(df)} rows")
summarize(df)

Loaded 88104 rows


{'rows': 88104,
 'images': 88104,
 'subjects': 7765,
 'visits_total': 16152,
 'visits_per_subject_median': 1.0,
 'visits_per_subject_max': 7,
 'subjects_longitudinal': 3652,
 'modalities': {'MRI': 37343, 'DTI': 30377, 'SPECT': 20384},
 'image_types': {'Original': 60040, 'Pre-processed': 28064},
 'distinct_descriptions': 3260,
 'date_range': ('1970-01-01', '2026-07-21'),
 'ignored': 756,
 'categories': {None: 40089,
  'anat/T1w': 17387,
  'dwi': 15829,
  'anat/T2w': 6115,
  'anat/FLAIR': 4290,
  'func': 3633,
  'anat/T2starw': 5},
 'uncategorized': 40089}

If `summary["uncategorized"]` is more than a handful, some Descriptions in
this export aren't covered yet. List them, decide (by hand, using the study
docs / `describe_sequences` counts below) whether each is a usable
acquisition or junk, and add it to `ppmi_description_categories.json` or
`ppmi_ignored_descriptions.csv`
(`src/mri_prep/datasets/`) -- then re-run the two cells above
(`reload_description_resources()` first, so the edit is picked up without
restarting the kernel).

In [3]:
uncategorized = df[df["category"].isna() & ~df["ignored"]]
describe_sequences(uncategorized)

,Modality,Description,scans,subjects,first,last
0,SPECT,Reconstructed DaTSCAN,3943,3530,2021-06-10,2024-11-29
1,SPECT,TOMO,1822,1248,2011-01-27,2026-07-02
2,DTI,EigenVal0-EPI <- DTI_gated,834,319,2010-11-09,2017-08-16
3,DTI,EigenVal2-EPI <- DTI_gated,834,319,2010-11-09,2017-08-16
4,DTI,EigenVectors-EPI <- DTI_gated,834,319,2010-11-09,2017-08-16
...,...,...,...,...,...,...
2674,MRI,T2 in corrected EPI space for I365117 <- Axial...,1,1,2013-02-04,2013-02-04
2675,MRI,T2 in corrected EPI space for I365156 <- Axial...,1,1,2012-11-06,2012-11-06
2676,MRI,T2 in corrected EPI space for I365162 <- Axial...,1,1,2013-02-01,2013-02-01
2677,MRI,T2 in corrected EPI space for I365199 <- Axial...,1,1,2012-08-27,2012-08-27


In [4]:
# After editing the curation files:
# reload_description_resources()
# df = adapter.annotate_categories(load_ida_search(search_csv))
# summarize(df)

## 2. Load clinical data

Two genuinely different tables -- pick one (or merge both, on `PATNO`/
`EVENT_ID`, before joining with the image rows below):

- **PPMI Curated Data Cut** (`data/PPMI/study/PPMI_Curated_Data_Cut_Public_*.xlsx`):
  PPMI's own pre-merged, one-row-per-visit summary table. What this notebook
  used historically, and what produced the existing
  `data/PPMI/tabular/ppmi_imaging_metadata.csv`.
- **`mri_prep.tabular.merge.merge_dataset_tables`** over the raw
  per-instrument CSVs in `data/PPMI/study/` (`mri-prep tabular merge`): finer
  grained, includes variables not in the curated cut, at the cost of manual
  column selection.

In [5]:
curated_xlsx = sorted(config.paths.csv_root.glob("PPMI_Curated_Data_Cut_Public_*.xlsx"))
assert curated_xlsx, "No PPMI_Curated_Data_Cut_Public_*.xlsx found under data/PPMI/study/"
curated_xlsx = curated_xlsx[-1]
print(f"Using curated data cut: {curated_xlsx.name}")

clinical_df = pd.read_excel(curated_xlsx)
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)
print(f"Clinical rows: {len(clinical_df)}, subjects: {clinical_df['PATNO'].nunique()}")

from mri_prep.datasets.ppmi import COHORT_MAP, PRIMDIAG_MAP

clinical_df["PRIMDIAG_DESC"] = clinical_df["PRIMDIAG"].map(PRIMDIAG_MAP)
clinical_df["COHORT_DESC"] = clinical_df["COHORT"].map(COHORT_MAP)

Using curated data cut: PPMI_Curated_Data_Cut_Public_20260511.xlsx


Clinical rows: 19450, subjects: 4788


## 3. Filter to usable image rows

`filter_images` drops rows flagged `ignored` by default; add any other
criteria as plain pandas (date range, `Type == "Original"` to exclude
derived/reconstructed images, ...) -- see AGENTS.md on why this stays hand
filtering rather than a query DSL.

In [6]:
images_df = filter_images(df, drop_ignored=True)
if "Type" in images_df.columns:
    images_df = images_df[(images_df["Type"] == "Original") | images_df["Type"].isna()]

images_df["PATNO"] = images_df["Subject ID"].astype(str)
from mri_prep.datasets.ppmi import VISIT_TO_EVENT_ID

images_df["EVENT_ID"] = images_df["Visit"].map(VISIT_TO_EVENT_ID)
print(f"Usable image rows: {len(images_df)}, subjects: {images_df['PATNO'].nunique()}")

Usable image rows: 59284, subjects: 7760


## 4. Join clinical + image rows

Outer join on `(PATNO, EVENT_ID)` to keep both clinical-only visits (no
imaging) and imaging-only rows (no matching clinical visit) -- filter either
out afterwards if you want a stricter cohort.

In [7]:
merged = pd.merge(clinical_df, images_df, how="outer", on=["PATNO", "EVENT_ID"])
print(f"Merged rows: {len(merged)}, subjects: {merged['PATNO'].nunique()}")

merged = merged.drop_duplicates(subset=["PATNO", "EVENT_ID", "Image ID"], keep="first")
assert not merged.duplicated(["PATNO", "EVENT_ID", "Image ID"]).any()

Merged rows: 69508, subjects: 7916


## 5. Save

Writes to `extra.merged_csv` from `configs/datasets/ppmi.yaml` -- the path
`mri_prep.bids.convert.convert_to_bids` reads.

In [8]:
out_path = REPO_ROOT / config.extra["merged_csv"]
out_path.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(out_path, index=False)
print(f"Saved {len(merged)} rows to {out_path}")

print(merged["BIDS_Modality"].value_counts(dropna=False))
print(merged["Advanced_Modality"].value_counts(dropna=False))

Saved 69508 rows to /home/falconnier/Documents/mri-preprocessing/data/PPMI/tabular/ppmi_imaging_metadata.csv
BIDS_Modality
anat    27797
dwi     15829
None    12025
NaN     10224
func     3633
Name: count, dtype: int64
Advanced_Modality
None       31487
T1w        17387
NaN        10224
T2w         6115
FLAIR       4290
T2starw        5
Name: count, dtype: int64


## Next

- `mri-prep ida image-ids --csv <your filtered selection>.csv --out ids.txt`
  to build the download collection for a chosen subset (see README
  "Selecting a cohort" step 5).
- `mri-prep bids convert --dataset ppmi` once the raw images are downloaded
  and flattened (README "Processing images" steps 1 and 3).
- `notebooks/ppmi_cohort_exploration.ipynb` for descriptive analysis of the
  table saved above (demographics, longitudinal coverage, modality overlap).